# #P y problemas de conteo

Este cuaderno explora la complejidad de contar soluciones:
el permanente de matrices, coloraciones de grafos y aproximación FPRAS.

**Artículo relacionado:** `04/11-sharp-p-y-conteo.md`

**Ejercicio resuelto:** `ejercicios/resueltos/` (no existe aún para este tema)

In [ ]:
import math
import random
import numpy as np
from itertools import permutations
from collections import defaultdict

## 1. El permanente de una matriz

$$\text{perm}(A) = \sum_{\sigma \in S_n} \prod_{i=1}^n A_{i,\sigma(i)}$$

El permanente cuenta matchings perfectos en el grafo bipartito de $A$.

### Ejercicio 1.1
Calcula el permanente de la siguiente matriz 3×3 **a mano** y verifica con la función:

In [ ]:
def permanent_naive(A):
    """Calcula el permanente por fuerza bruta: O(n! · n)."""
    n = len(A)
    total = 0
    for sigma in permutations(range(n)):
        product = 1
        for i, j in enumerate(sigma):
            product *= A[i][j]
        total += product
    return total


# Matriz de ejemplo del artículo
A = [
    [1, 0, 1],
    [1, 1, 0],
    [0, 1, 1]
]

perm_A = permanent_naive(A)
print(f"Permanente de A = {perm_A}")
print()
print("Interpretación: número de matchings perfectos en el grafo bipartito")
print("con aristas {(1,1),(1,3),(2,1),(2,2),(3,2),(3,3)}")

# TODO del estudiante:
# Calcula a mano las 3! = 6 permutaciones y suma los productos:
# σ=(0,1,2): A[0][0]·A[1][1]·A[2][2] = ?
# σ=(0,2,1): A[0][0]·A[1][2]·A[2][1] = ?
# σ=(1,0,2): A[0][1]·A[1][0]·A[2][2] = ?
# σ=(1,2,0): A[0][1]·A[1][2]·A[2][0] = ?
# σ=(2,0,1): A[0][2]·A[1][0]·A[2][1] = ?
# σ=(2,1,0): A[0][2]·A[1][1]·A[2][0] = ?
# SUMA = ?

### Ejercicio 1.2
El permanente vs el determinante: misma fórmula, pero el determinante usa signos.
Verifica que el determinante se puede calcular en $O(n^3)$ pero el permanente requiere $O(2^n \cdot n)$ con el mejor algoritmo conocido (Ryser).

In [ ]:
def permanent_ryser(A):
    """
    Algoritmo de Ryser para el permanente: O(2^n · n).
    Mucho más rápido que n! para n grande.
    """
    n = len(A)
    total = 0
    # Iterar sobre todos los subconjuntos de columnas
    for S in range(1, 2**n):
        cols = [j for j in range(n) if (S >> j) & 1]
        k = len(cols)
        # Producto de sumas de filas sobre S
        row_sums = [sum(A[i][j] for j in cols) for i in range(n)]
        prod = 1
        for rs in row_sums:
            prod *= rs
        # Coeficiente de inclusión-exclusión
        sign = (-1)**(n - k)
        total += sign * prod
    return (-1)**n * total


# Comparación de tiempo para matrices de distinto tamaño
import time

print(f"{'n':<5} {'perm_naive (s)':<18} {'perm_ryser (s)':<18} {'det numpy (s)':<15} {'perm':<10}")
print('-' * 65)
for n in [3, 4, 5, 6, 8, 10]:
    random.seed(42)
    A_n = [[random.randint(0, 1) for _ in range(n)] for _ in range(n)]
    
    t0 = time.time()
    if n <= 8:
        pn = permanent_naive(A_n)
        t_naive = time.time() - t0
    else:
        pn = '(omitido)'
        t_naive = float('nan')
    
    t0 = time.time()
    pr = permanent_ryser(A_n)
    t_ryser = time.time() - t0
    
    t0 = time.time()
    det = np.linalg.det(np.array(A_n, dtype=float))
    t_det = time.time() - t0
    
    print(f"{n:<5} {t_naive:<18.6f} {t_ryser:<18.6f} {t_det:<15.6f} {pr}")

## 2. Contar coloraciones de grafos

In [ ]:
def count_colorings_exact(graph, k, n_vertices):
    """
    Cuenta exactamente el número de k-coloraciones propias de un grafo.
    Fuerza bruta: O(k^n · m) donde m = número de aristas.
    """
    count = 0
    for coloring in range(k**n_vertices):
        # Decodificar la coloración
        colors = []
        c = coloring
        for _ in range(n_vertices):
            colors.append(c % k)
            c //= k
        # Verificar que es válida
        valid = all(colors[u] != colors[v] for u, v in graph)
        if valid:
            count += 1
    return count


# Grafo completo K3 (triángulo)
K3 = [(0,1), (1,2), (0,2)]

print("Grafo K3 (triángulo):")
for k in range(1, 6):
    count = count_colorings_exact(K3, k, 3)
    # Polinomio cromático de K3: P(K3, k) = k(k-1)(k-2)
    polinomio = k * (k-1) * (k-2) if k >= 3 else 0
    print(f"  k={k}: {count} coloraciones (polinomio cromático: {polinomio})")

print()

# Grafo C4 (cuadrado)
C4 = [(0,1), (1,2), (2,3), (0,3)]
print("Grafo C4 (cuadrado):")
for k in range(1, 6):
    count = count_colorings_exact(C4, k, 4)
    # Polinomio cromático de C4: (k-1)^4 + (k-1)
    polinomio = (k-1)**4 + (k-1)
    print(f"  k={k}: {count} coloraciones (polinomio cromático: {polinomio})")

### Ejercicio 2.1
El **polinomio cromático** $P(G, k)$ da el número de $k$-coloraciones propias.
Para K_n (grafo completo con $n$ vértices), $P(K_n, k) = k(k-1)(k-2)\cdots(k-n+1)$.

**Completa:** ¿cuántas 3-coloraciones tiene $K_4$? ¿Y $K_5$?

In [ ]:
# Tu solución aquí:
# K4 con k=3 colores: P(K4, 3) = ?
# K5 con k=3 colores: P(K5, 3) = ?

# Pista: aplica la fórmula k(k-1)(k-2)...(k-n+1)
# o usa el código de conteo exacto (cuidado: K5 tiene 5 vértices, puede ser lento)

# Solución de referencia (comentada):
# K4_edges = [(i,j) for i in range(4) for j in range(i+1,4)]
# print(f"K4 con 3 colores: {count_colorings_exact(K4_edges, 3, 4)}")
# print(f"Fórmula: 3·2·1·0 = {3*2*1*0}")
# print(f"K5 con 3 colores: 3·2·1·0·(-1) = {3*2*1*0*(-1)} → imposible (número negativo = 0 real)")

## 3. Aproximación FPRAS: estimación por muestreo

In [ ]:
def estimate_matchings_monte_carlo(n, edges, n_samples=10000):
    """
    Estimación Monte Carlo del número de matchings perfectos en un grafo bipartito.
    
    Estrategia: muestrear permutaciones aleatorias y verificar que forman matching.
    (Solo funciona bien cuando hay muchos matchings.)
    """
    random.seed(42)
    adj = set(edges)
    hits = 0
    
    for _ in range(n_samples):
        # Muestra una permutación aleatoria
        sigma = list(range(n))
        random.shuffle(sigma)
        # Verificar si es matching perfecto
        if all((i, sigma[i]) in adj for i in range(n)):
            hits += 1
    
    # El estimador: hits/n_samples ≈ perm(A) / n!
    estimated_perm = hits * math.factorial(n) / n_samples
    return estimated_perm, hits


# Grafo bipartito 3×3: cada vértice izquierdo conectado a todos los derechos
# → matriz de todos unos → permanente = n!
n = 4
edges_complete = [(i, j) for i in range(n) for j in range(n)]
A_complete = [[1]*n for _ in range(n)]

exact = permanent_ryser(A_complete)
estimated, hits = estimate_matchings_monte_carlo(n, edges_complete, n_samples=50000)

print(f"Grafo bipartito completo K_{{{n},{n}}}:")
print(f"Permanente exacto: {exact} = {n}!")
print(f"Estimación Monte Carlo ({50000} muestras): {estimated:.1f}")
print(f"Hits: {hits}")
print(f"Error relativo: {abs(estimated - exact)/exact * 100:.2f}%")
print()

# Con matriz dispersa, pocos matchings → muestreo ineficiente
A_sparse = [
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 1]
]
edges_sparse = [(i, j) for i in range(n) for j in range(n) if A_sparse[i][j]]

exact_s = permanent_ryser(A_sparse)
estimated_s, hits_s = estimate_matchings_monte_carlo(n, edges_sparse, n_samples=50000)

print(f"Grafo bipartito disperso:")
print(f"Permanente exacto: {exact_s}")
print(f"Estimación Monte Carlo: {estimated_s:.2f}")
print(f"Error relativo: {abs(estimated_s - exact_s)/max(exact_s,1) * 100:.2f}%")
print(f"Nota: con grafos dispersos, el muestreo aleatorio de permutaciones es ineficiente.")
print(f"Los algoritmos FPRAS modernos (Jerrum-Sinclair-Vigoda) usan MCMC sobre matchings.")

## 4. El teorema de Toda: counting es más poderoso que NP

### Ejercicio 4.1 — Conceptual

Responde las siguientes preguntas sobre #P y el teorema de Toda:

In [ ]:
# Preguntas conceptuales — escribe tus respuestas aquí:

preguntas = {
    "1": """¿Por qué decidir si el permanente es > 0 es fácil (P), 
          pero calcularlo exactamente es #P-completo?""",
    
    "2": """El teorema de Toda dice PH ⊆ P^#P. 
          ¿Qué implica esto sobre la relación entre contar y decidir?""",
    
    "3": """Si encontrásemos un algoritmo en tiempo polinomial para #SAT,
          ¿qué implicaría sobre P vs NP?""",
}

for k, q in preguntas.items():
    print(f"Pregunta {k}: {q}")
    print("Tu respuesta: [escribe aquí]")
    print()

# Soluciones de referencia:
# 1. perm(A)>0 ⟺ existe un matching perfecto, que se verifica en P (matchings bipartitos).
#    Calcular el valor exacto requiere sumar sobre todos los matchings, que es #P-completo.
# 2. El conteo (#P) es al menos tan poderoso como toda la jerarquía polinómica:
#    una sola consulta a un oráculo #P permite resolver cualquier problema de PH.
# 3. P^#SAT = P (si #SAT ∈ FP), y PH ⊆ P → PH = P, lo que implicaría P = NP.